# Lab 07 — Graph Visualization (Solution, chi tiết)

## 1) Graph fundamentals\nMục tiêu: phân biệt node/edge, directed/undirected, weighted/unweighted và đọc thống kê cơ bản.

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# TODO: hoàn thiện cell này theo đề bài



In [ ]:

# TODO: hoàn thiện cell này theo đề bài



## 2) Data-to-graph construction\nMục tiêu: từ bảng quan hệ tạo graph mới (không chỉ dùng graph có sẵn).

In [ ]:

# TODO: hoàn thiện cell này theo đề bài



## 3) Layout methods comparison\nMục tiêu: thấy layout khác nhau dẫn tới cảm nhận pattern khác nhau.

In [ ]:

# TODO: hoàn thiện cell này theo đề bài



## 4) Visual encoding choices\nMục tiêu: node size theo degree, color theo community, edge width theo weight.

In [ ]:

# TODO: hoàn thiện cell này theo đề bài



## 5) Core graph tasks (insight bắt buộc)

In [ ]:

# TODO: hoàn thiện cell này theo đề bài



In [ ]:

# TODO: hoàn thiện cell này theo đề bài



## 6) Large-graph handling\nMục tiêu: giảm rối bằng filtering/subgraph.

In [ ]:

# TODO: hoàn thiện cell này theo đề bài



## 7) Interpretation and limitations

In [ ]:

# TODO: hoàn thiện cell này theo đề bài



## Reflection\n- Vì sao bạn chọn spring/kamada cho graph chính?\n- Nếu dữ liệu của bạn có hướng (directed), insight nào sẽ thay đổi?\n- Hãy nêu một rủi ro diễn giải sai khi chỉ nhìn layout mà không xem metric.

## 8) Force-directed graph (nhấn mạnh)
Force-directed layout (spring) mô phỏng lực hút/đẩy để đưa các node liên quan gần nhau, phù hợp cho việc nhìn community trực quan.

In [ ]:
plt.figure(figsize=(8, 6))
pos_force = nx.spring_layout(G, seed=7, k=0.35, iterations=100)
nx.draw_networkx(
    G,
    pos=pos_force,
    with_labels=False,
    node_size=[80 + 35 * deg[n] for n in G.nodes()],
    node_color=[node2comm[n] for n in G.nodes()],
    cmap=plt.cm.Set2,
    edge_color="gray",
    alpha=0.9,
)
plt.title("Force-directed (spring) layout")
plt.axis("off")
plt.show()

## 9) Multi-level technique (coarsen -> refine)
Ý tưởng: gom node theo community thành **super-node** để xem cấu trúc macro, sau đó drill-down vào community cần phân tích.

In [ ]:
# Build community-level (coarsened) graph
comm_map = {n: cid for cid, comm in enumerate(communities) for n in comm}
G_coarse = nx.Graph()
for cid in set(comm_map.values()):
    G_coarse.add_node(cid, size=sum(1 for n in comm_map if comm_map[n] == cid))

for u, v in G.edges():
    cu, cv = comm_map[u], comm_map[v]
    if cu == cv:
        continue
    if G_coarse.has_edge(cu, cv):
        G_coarse[cu][cv]["weight"] += 1
    else:
        G_coarse.add_edge(cu, cv, weight=1)

coarse_info = pd.DataFrame(
    {
        "coarse_nodes": [G_coarse.number_of_nodes()],
        "coarse_edges": [G_coarse.number_of_edges()],
        "original_nodes": [G.number_of_nodes()],
        "original_edges": [G.number_of_edges()],
    }
)
coarse_info

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Coarsened view (collapse)
pos_c = nx.spring_layout(G_coarse, seed=2)
coarse_sizes = [900 * G_coarse.nodes[n]["size"] for n in G_coarse.nodes()]
coarse_widths = [1 + 0.25 * G_coarse[u][v]["weight"] for u, v in G_coarse.edges()]
nx.draw_networkx(G_coarse, pos=pos_c, node_size=coarse_sizes, width=coarse_widths, ax=axes[0])
axes[0].set_title("Collapsed community graph")
axes[0].axis("off")

# Expand one community (drill-down)
target_comm = max(set(comm_map.values()), key=lambda c: sum(1 for n in comm_map if comm_map[n] == c))
nodes_target = [n for n in G.nodes() if comm_map[n] == target_comm]
G_expand = G.subgraph(nodes_target).copy()
pos_e = nx.spring_layout(G_expand, seed=2)
nx.draw_networkx(G_expand, pos=pos_e, with_labels=True, node_size=220, ax=axes[1])
axes[1].set_title(f"Expanded community {target_comm}")
axes[1].axis("off")

plt.tight_layout(); plt.show()

## 10) Sampling techniques cho large graphs
- Random Sampling
- Degree-Based Sampling
- Ego-Network Sampling
- Community Sampling

In [ ]:
rng = np.random.default_rng(42)
all_nodes = list(G.nodes())

# 1) Random sampling
rand_nodes = rng.choice(all_nodes, size=12, replace=False).tolist()
G_rand = G.subgraph(rand_nodes).copy()

# 2) Degree-based sampling
deg_sorted = sorted(G.degree, key=lambda x: x[1], reverse=True)
deg_nodes = [n for n, _ in deg_sorted[:12]]
G_deg = G.subgraph(deg_nodes).copy()

# 3) Ego-network sampling
ego_center = deg_sorted[0][0]
G_ego = nx.ego_graph(G, ego_center, radius=1)

# 4) Community sampling (largest community)
largest_comm = max(communities, key=len)
G_comm = G.subgraph(list(largest_comm)).copy()

pd.DataFrame(
    {
        "technique": ["random", "degree_based", "ego_network", "community"],
        "nodes": [G_rand.number_of_nodes(), G_deg.number_of_nodes(), G_ego.number_of_nodes(), G_comm.number_of_nodes()],
        "edges": [G_rand.number_of_edges(), G_deg.number_of_edges(), G_ego.number_of_edges(), G_comm.number_of_edges()],
    }
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
plots = [
    (G_rand, "Random sampling"),
    (G_deg, "Degree-based sampling"),
    (G_ego, f"Ego-network (center={ego_center})"),
    (G_comm, "Community sampling"),
]
for ax, (g, title) in zip(axes.ravel(), plots, strict=True):
    pos = nx.spring_layout(g, seed=1)
    nx.draw_networkx(g, pos=pos, with_labels=False, node_size=140, ax=ax)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 11) Circos layout + Edge bundling (xấp xỉ)
Dùng circular/circos layout để tận dụng không gian và dễ nhìn nhóm. Edge bundling đầy đủ cần thư viện chuyên dụng; ở đây mô phỏng bằng cách chỉ giữ inter-community edges để giảm rối.

In [ ]:
# Sort nodes by community for a cleaner circos-like ring
nodes_sorted = sorted(G.nodes(), key=lambda n: (comm_map[n], n))
G_sorted = G.subgraph(nodes_sorted).copy()
pos_ring = nx.circular_layout(nodes_sorted)

# Approximate "bundling": keep only cross-community edges
cross_edges = [(u, v) for u, v in G_sorted.edges() if comm_map[u] != comm_map[v]]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

nx.draw_networkx(
    G_sorted,
    pos=pos_ring,
    with_labels=False,
    node_size=120,
    node_color=[comm_map[n] for n in nodes_sorted],
    cmap=plt.cm.tab20,
    edge_color="lightgray",
    ax=axes[0],
)
axes[0].set_title("Circos-like circular layout")
axes[0].axis("off")

nx.draw_networkx_nodes(
    G_sorted,
    pos=pos_ring,
    node_size=120,
    node_color=[comm_map[n] for n in nodes_sorted],
    cmap=plt.cm.tab20,
    ax=axes[1],
)
nx.draw_networkx_edges(
    G_sorted,
    pos=pos_ring,
    edgelist=cross_edges,
    alpha=0.55,
    width=1.2,
    edge_color="tab:blue",
    ax=axes[1],
)
axes[1].set_title("Edge-bundling approximation (inter-community edges only)")
axes[1].axis("off")

plt.tight_layout(); plt.show()

## Reflection mở rộng
- Force-directed giúp thấy cụm, nhưng vị trí tuyệt đối không mang ý nghĩa metric.
- Multi-level (collapse/expand) giúp đọc macro rồi đi sâu micro.
- Sampling khác nhau sẽ làm nổi bật các khía cạnh khác nhau của mạng.
- Circos + bundling phù hợp storytelling cho mạng lớn, nhưng có thể làm mất chi tiết cục bộ.